# BricksVault: fine-tune SigLIP for photo scans

This notebook teaches the SigLIP image model that the photo scanner uses as a second opinion to tell LEGO sets apart better, then measures whether it really helps before anything changes in the app.

**How to run**
1. **Runtime → Change runtime type → A100 GPU** (H100 or L4 also work), then Save.
2. **Runtime → Run all.** Allow Google Drive access when asked, so downloads and results survive a disconnect.
3. Wait about 1 to 1.5 hours. The first run downloads about 20,000 catalog images (about 15 minutes). Re-runs reuse them.
4. At the end, copy the **summary box** (last cell) into the BricksVault thread.

**What it does**
- Downloads every catalog set image (Rebrickable CDN) and the 78 real-world test photos used for the scanner tests.
- Measures the current model first (same numbers as the app today), with Brickognize's saved answers for those photos.
- Fine-tunes SigLIP on catalog images turned into "fake real photos" (random backgrounds, angles, crops, lighting). Training batches are full of look-alike sets, so it learns the small differences. The test photos are never used for training.
- Measures again and prints a side-by-side table: how many scans the app would answer right, answer wrong, or hand to the AI.
- Exports the new model in the exact format the scanner service loads, checks it gives the same numbers as PyTorch, and saves it to Drive (`MyDrive/bricksvault-siglip/`).

Nothing is uploaded anywhere and no keys are needed. The model is only switched into the app if the numbers say it is better.

In [ ]:
# Settings. The defaults are what Claude recommends; no need to change anything.
import os
QUICK = os.environ.get('SIGLIP_QUICK') == '1'   # tiny smoke run (used for testing the notebook itself)

BASE_MODEL = 'google/siglip-base-patch16-224'    # same weights as Xenova/siglip-base-patch16-224 used in the app
ONNX_TEMPLATE = 'Xenova/siglip-base-patch16-224' # config files for the exported model
API = 'https://brickvault-api.zhydenko.workers.dev'
RAW = os.environ.get('SIGLIP_RAW', 'https://raw.githubusercontent.com/yarik1390/MyBricks/main/scripts/scan-eval')

EPOCHS = 1 if QUICK else 8
LR = 2e-5
WEIGHT_DECAY = 0.05
FREEZE_LAYERS = 4          # keep the first 4 of 12 transformer layers as they are
TEMPERATURE = 0.05
NEIGHBOURS = 15            # look-alike sets added next to each random set in a batch
MAX_SETS = 300 if QUICK else None
MAX_TEST_PHOTOS = 12 if QUICK else None
ACCEPT_MARGIN = 0.20       # same as SIGLIP_ACCEPT_MARGIN in worker/src/lib/siglip.ts
BK_SCORE_MIN, BK_MARGIN_MIN = 0.60, 0.10  # Brickognize gate in worker/src/lib/brickognize.ts
SEED = 1390

In [ ]:
# Install and set up storage.
import subprocess, sys
def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)
try:
    import onnx, onnxruntime  # noqa: F401
except ImportError:
    pip('onnx', 'onnxruntime')
import transformers
if tuple(int(x) for x in transformers.__version__.split('.')[:2]) < (4, 40):
    pip('transformers>=4.40')

WORK = '/content/bricksvault-siglip' if os.path.isdir('/content') else os.path.abspath('bricksvault-siglip')
if not QUICK:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        WORK = '/content/drive/MyDrive/bricksvault-siglip'
    except Exception as e:
        print('Google Drive not available, using local disk only:', e)
# Images live on the fast local disk; a single archive in the working folder (Drive) makes re-runs skip the download.
LOCAL = '/content/local' if os.path.isdir('/content') else os.path.join(WORK, 'local')
CACHE = os.path.join(LOCAL, 'catalog')
OUT = os.path.join(WORK, 'output')
ARCHIVE = os.path.join(WORK, 'catalog-images.tar')
os.makedirs(CACHE, exist_ok=True); os.makedirs(OUT, exist_ok=True)
import tarfile
if os.path.exists(ARCHIVE) and len(os.listdir(CACHE)) == 0:
    print('Restoring saved catalog images...')
    with tarfile.open(ARCHIVE) as tf: tf.extractall(LOCAL)
print('Working folder:', WORK)

In [ ]:
# Catalog set list and test photos.
import json, random, time, io
import requests
from concurrent.futures import ThreadPoolExecutor
UA = {'User-Agent': 'BricksVault-training/1.0 (+https://bricksvault.app)'}
random.seed(SEED)

def get_json(url, tries=4):
    if url.startswith('/'): return json.load(open(url))
    for i in range(tries):
        try:
            r = requests.get(url, headers=UA, timeout=60)
            if r.ok: return r.json()
        except Exception: pass
        time.sleep(2 * (i + 1))
    raise RuntimeError('could not load ' + url)

sets, page, pages = [], 1, 1
while page <= pages:
    d = get_json(f'{API}/api/seo/sitemap-sets?page={page}')
    pages = d['pages']; sets += [s['set_num'] for s in d['sets']]; page += 1
tests = get_json(f'{RAW}/test-photos.json')
bk_rows = {r['url']: r['brickognize'] for r in get_json(f'{RAW}/brickognize-top5.json')}
if MAX_TEST_PHOTOS: tests = tests[:MAX_TEST_PHOTOS]
test_sets = sorted({t['set'] for t in tests})
if MAX_SETS:
    sets = test_sets + random.sample([s for s in sets if s not in test_sets], MAX_SETS - len(test_sets))
sets = sorted(set(sets) | set(test_sets))
print(f'{len(sets):,} catalog sets, {len(tests)} test photos of {len(test_sets)} sets')

In [ ]:
# Download images (resized to 384 px, cached in the working folder).
from PIL import Image, ImageOps
def cache_path(s): return os.path.join(CACHE, s + '.jpg')

def fetch_image(url, dest, tries=3):
    if os.path.exists(dest): return True
    if os.path.exists(dest + '.missing'): return False
    for i in range(tries):
        try:
            r = requests.get(url, headers=UA, timeout=30)
            if r.status_code == 404:
                open(dest + '.missing', 'w').close(); return False
            if r.ok:
                im = ImageOps.exif_transpose(Image.open(io.BytesIO(r.content))).convert('RGB')
                im.thumbnail((384, 384), Image.BICUBIC)
                im.save(dest, quality=92); return True
        except Exception: pass
        time.sleep(1.5 * (i + 1))
    return False

t0 = time.time()
with ThreadPoolExecutor(16) as ex:
    ok = list(ex.map(lambda s: fetch_image(f'https://cdn.rebrickable.com/media/sets/{s}.jpg', cache_path(s)), sets))
catalog = [s for s, k in zip(sets, ok) if k]
print(f'catalog images: {len(catalog):,} of {len(sets):,} ({time.time() - t0:.0f}s)')
if not QUICK and time.time() - t0 > 60:   # new downloads: save them for next time
    with tarfile.open(ARCHIVE + '.tmp', 'w') as tf: tf.add(CACHE, arcname='catalog')
    os.replace(ARCHIVE + '.tmp', ARCHIVE)

TEST_DIR = os.path.join(LOCAL, 'test-photos'); os.makedirs(TEST_DIR, exist_ok=True)
for i, t in enumerate(tests): t['path'] = os.path.join(TEST_DIR, f'{i:03d}.jpg')
with ThreadPoolExecutor(8) as ex:
    ok = list(ex.map(lambda t: fetch_image(t['url'], t['path']), tests))
tests = [t for t, k in zip(tests, ok) if k]
print(f'test photos: {len(tests)}')

In [ ]:
# Model, embedding helpers and the scanner's decision rule (copied from the app).
import numpy as np, torch, torch.nn.functional as F
from transformers import SiglipVisionModel, AutoImageProcessor
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu' and not QUICK:
    raise SystemExit('No GPU: choose Runtime → Change runtime type → A100 GPU, then Run all again.')
amp = torch.bfloat16 if device == 'cuda' else torch.float32
processor = AutoImageProcessor.from_pretrained(BASE_MODEL)

class ImageFiles(torch.utils.data.Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        im = Image.open(self.paths[i]).convert('RGB')
        return processor(images=im, return_tensors='pt')['pixel_values'][0]

@torch.no_grad()
def embed_paths(model, paths, bs=256):
    model.eval()
    dl = torch.utils.data.DataLoader(ImageFiles(paths), batch_size=bs, num_workers=min(8, os.cpu_count() or 2))
    out = []
    for x in dl:
        with torch.autocast(device_type=device, dtype=amp, enabled=device == 'cuda'):
            v = model(pixel_values=x.to(device)).pooler_output
        out.append(F.normalize(v.float(), dim=-1).cpu())
    return torch.cat(out)

canon = lambda s: s if '-' in s and s.rsplit('-', 1)[1].isdigit() else s + '-1'
bare = lambda s: canon(s).rsplit('-', 1)[0]

def fuse(preds, cand_sims, sig_top):
    sims = list(cand_sims.values()) + [x[1] for x in sig_top]
    if not preds or not sims: return None
    lo = min(sims); by = {}
    for p in preds:
        i = canon(p['id'])
        if bare(i) not in by: by[bare(i)] = (i, p['score'], cand_sims.get(i, lo))
    for i, sim in sig_top:
        i = canon(i)
        if bare(i) not in by: by[bare(i)] = (i, 0.0, sim)
    ranked = sorted(((i, b + (s - lo)) for i, b, s in by.values()), key=lambda x: -x[1])
    return ranked, (ranked[0][1] - ranked[1][1] if len(ranked) > 1 else ranked[0][1])

def evaluate(model, label):
    gal = embed_paths(model, [cache_path(s) for s in catalog])
    q = embed_paths(model, [t['path'] for t in tests])
    sims = q @ gal.T
    idx = {s: i for i, s in enumerate(catalog)}
    top1 = top5 = 0; rows = []
    for k, t in enumerate(tests):
        order = torch.topk(sims[k], 5).indices.tolist()
        names = [catalog[i] for i in order]
        top1 += bare(names[0]) == bare(t['set']); top5 += any(bare(n) == bare(t['set']) for n in names)
        preds = bk_rows.get(t['url'], [])
        cand = {canon(p['id']): float(sims[k, idx[canon(p['id'])]]) for p in preds if canon(p['id']) in idx}
        f = fuse(preds, cand, [(catalog[i], float(sims[k, i])) for i in order])
        sure = bool(preds) and preds[0]['score'] >= BK_SCORE_MIN and (len(preds) < 2 or preds[0]['score'] - preds[1]['score'] >= BK_MARGIN_MIN)
        rows.append(dict(set=t['set'], kind=t['kind'], sure=sure, bk=canon(preds[0]['id']) if preds else None,
                         fused=f[0][0][0] if f else None, margin=f[1] if f else 0.0))
    def outcome(margin):
        right = wrong = ai = 0
        for r in rows:
            if r['sure']: ans = r['bk']
            elif r['fused'] and r['margin'] >= margin: ans = r['fused']
            else: ai += 1; continue
            if bare(ans) == bare(r['set']): right += 1
            else: wrong += 1
        return right, wrong, ai
    fused_first = sum(1 for r in rows if r['fused'] and bare(r['fused']) == bare(r['set']))
    res = dict(label=label, photos=len(tests), siglip_top1=top1, siglip_top5=top5, fused_first=fused_first,
               app=outcome(ACCEPT_MARGIN), sweep={m: outcome(m) for m in (0.05, 0.10, 0.15, 0.20, 0.25, 0.30)}, rows=rows)
    r, w, a = res['app']
    print(f"{label}: SigLIP alone top-1 {top1}/{len(tests)}, top-5 {top5}/{len(tests)}; "
          f"combined first answer {fused_first}/{len(tests)}; app at margin {ACCEPT_MARGIN}: {r} right, {w} wrong, {a} to AI")
    return res, gal

base_model = SiglipVisionModel.from_pretrained(BASE_MODEL).to(device)
baseline, base_gal = evaluate(base_model, 'Current model')

In [ ]:
# Training data: catalog images turned into realistic photos.
import torchvision.transforms as T
from PIL import ImageFilter
MEAN, STD = processor.image_mean, processor.image_std
to_tensor = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])
photo_aug = T.Compose([
    T.RandomResizedCrop(224, scale=(0.35, 1.0), ratio=(0.6, 1.6), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomApply([T.RandomRotation(20, interpolation=T.InterpolationMode.BILINEAR, fill=0)], p=0.5),
    T.RandomPerspective(0.3, p=0.4),
    T.ColorJitter(0.4, 0.4, 0.35, 0.05),
    T.RandomGrayscale(0.05),
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.3),
])
clean = T.Compose([T.Resize((224, 224), interpolation=T.InterpolationMode.BICUBIC)])

def random_background(size, others):
    w, h = size; r = random.random()
    if r < 0.45:   # a blurred, enlarged piece of another catalog image (shelf/table clutter)
        bg = Image.open(cache_path(random.choice(others))).convert('RGB')
        return bg.resize((w * 2, h * 2)).crop((w // 2, h // 2, w // 2 + w, h // 2 + h)).filter(ImageFilter.GaussianBlur(random.uniform(2, 8)))
    if r < 0.75:   # plain wall / table colour with a soft gradient
        c1 = np.array([random.randint(30, 235) for _ in range(3)], dtype=np.float32)
        c2 = np.clip(c1 + np.random.uniform(-60, 60, 3), 0, 255)
        g = np.linspace(0, 1, h, dtype=np.float32)[:, None, None]
        return Image.fromarray((c1 * (1 - g) + c2 * g).repeat(w, 1).astype(np.uint8))
    noise = np.random.randint(0, 255, (max(h // 16, 1), max(w // 16, 1), 3), dtype=np.uint8)  # carpet-like texture
    return Image.fromarray(noise).resize((w, h), Image.BICUBIC).filter(ImageFilter.GaussianBlur(1.5))

def as_photo(im, others):
    if random.random() < 0.8:   # replace the white catalog background
        a = np.asarray(im).astype(np.int16)
        mask = Image.fromarray(((a.min(-1) < 235) * 255).astype(np.uint8)).filter(ImageFilter.MaxFilter(3)).filter(ImageFilter.GaussianBlur(1))
        bg = random_background(im.size, others); bg.paste(im, (0, 0), mask); im = bg
    return photo_aug(im)

class Pairs(torch.utils.data.Dataset):
    def __len__(self): return len(catalog)
    def __getitem__(self, i):
        im = Image.open(cache_path(catalog[i])).convert('RGB')
        return to_tensor(as_photo(im, catalog)), to_tensor(clean(im)), i

# Batches full of look-alikes: each random set brings its nearest neighbours by the current model.
knn = []
for c in range(0, len(catalog), 4096):
    s = base_gal[c:c + 4096] @ base_gal.T
    knn.append(torch.topk(s, min(33, len(catalog)), dim=1).indices[:, 1:])
knn = torch.cat(knn).tolist()

gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30 if device == 'cuda' else 0
BATCH = 16 if QUICK else (512 if gpu_gb >= 70 else 256 if gpu_gb >= 35 else 128)

class LookAlikeBatches(torch.utils.data.Sampler):
    def __len__(self): return max(1, len(catalog) // BATCH)
    def __iter__(self):
        for _ in range(len(self)):
            batch, seen = [], set()
            while len(batch) < BATCH:
                seed = random.randrange(len(catalog))
                for j in [seed] + random.sample(knn[seed], min(NEIGHBOURS, len(knn[seed]))):
                    if j not in seen and len(batch) < BATCH: seen.add(j); batch.append(j)
            yield batch
print(f'batch size {BATCH}, {len(LookAlikeBatches())} steps per epoch, {EPOCHS} epochs')

In [ ]:
# Fine-tune.
torch.manual_seed(SEED)
model = SiglipVisionModel.from_pretrained(BASE_MODEL).to(device)
vm = getattr(model, 'vision_model', model)   # transformers 4.x nests it, 5.x does not
for p in vm.embeddings.parameters(): p.requires_grad = False
for layer in vm.encoder.layers[:FREEZE_LAYERS]:
    for p in layer.parameters(): p.requires_grad = False
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
dl = torch.utils.data.DataLoader(Pairs(), batch_sampler=LookAlikeBatches(), num_workers=min(10, os.cpu_count() or 2), persistent_workers=not QUICK)
total = EPOCHS * len(dl); warm = max(1, total // 20)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warm) * 0.5 * (1 + np.cos(np.pi * min(1, s / total))))

history, best = [], None
for epoch in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); losses = []
    for q, k, _ in dl:
        with torch.autocast(device_type=device, dtype=amp, enabled=device == 'cuda'):
            zq = F.normalize(model(pixel_values=q.to(device)).pooler_output.float(), dim=-1)
            zk = F.normalize(model(pixel_values=k.to(device)).pooler_output.float(), dim=-1)
        logits = zq @ zk.T / TEMPERATURE
        y = torch.arange(len(zq), device=device)
        loss = (F.cross_entropy(logits, y) + F.cross_entropy(logits.T, y)) / 2
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step()
        losses.append(loss.item())
    print(f'epoch {epoch}/{EPOCHS}: loss {np.mean(losses):.3f} ({time.time() - t0:.0f}s)')
    res, _ = evaluate(model, f'Epoch {epoch}')
    history.append({k: v for k, v in res.items() if k != 'rows'})
torch.save(model.state_dict(), os.path.join(OUT, 'siglip-finetuned.pt'))
finetuned = res

In [ ]:
# Side-by-side result.
def line(r):
    a = r['app']
    return (f"| {r['label']} | {r['siglip_top1']}/{r['photos']} | {r['siglip_top5']}/{r['photos']} | "
            f"{r['fused_first']}/{r['photos']} | {a[0]} | {a[1]} | {a[2]} |")
table = '\n'.join(['| Model | SigLIP alone top-1 | top-5 | Combined first answer | App: right | App: wrong | App: to AI |',
                   '|---|---|---|---|---|---|---|', line(baseline), line(dict(finetuned, label='Fine-tuned'))])
sweep = '\n'.join(['| Accept margin | Current (right/wrong/AI) | Fine-tuned (right/wrong/AI) |', '|---|---|---|'] +
                  [f"| {m:.2f} | {baseline['sweep'][m]} | {finetuned['sweep'][m]} |" for m in baseline['sweep']])
b, f = baseline['app'], finetuned['app']
better = f[0] > b[0] and f[1] <= b[1]
verdict = ('Fine-tuned model is BETTER: more right answers with no extra wrong ones.' if better else
           'Fine-tuned model is NOT better at the app setting; keep the current model.')
summary = f"""BricksVault SigLIP fine-tune result
GPU: {torch.cuda.get_device_name(0) if device == 'cuda' else 'cpu'}, {len(catalog):,} catalog sets, {len(tests)} test photos, {EPOCHS} epochs, batch {BATCH}
{verdict}

{table}

{sweep}
"""
with open(os.path.join(OUT, 'results.md'), 'w') as fh: fh.write(summary)
with open(os.path.join(OUT, 'results.json'), 'w') as fh:
    json.dump({'baseline': baseline, 'finetuned': finetuned, 'history': history,
               'settings': dict(EPOCHS=EPOCHS, LR=LR, FREEZE_LAYERS=FREEZE_LAYERS, TEMPERATURE=TEMPERATURE, BATCH=BATCH)}, fh, indent=1)
print(summary)

In [ ]:
# Export in the format the scanner service loads (transformers.js: onnx/vision_model.onnx).
import shutil
from huggingface_hub import hf_hub_download
MODEL_DIR = os.path.join(OUT, 'siglip-bricksvault')
os.makedirs(os.path.join(MODEL_DIR, 'onnx'), exist_ok=True)
for name in ('config.json', 'preprocessor_config.json'):
    shutil.copy(hf_hub_download(ONNX_TEMPLATE, name), os.path.join(MODEL_DIR, name))

class VisionOnly(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, pixel_values):
        o = self.m(pixel_values=pixel_values)
        return o.last_hidden_state, o.pooler_output

export_model = VisionOnly(model.float().cpu().eval())
onnx_path = os.path.join(MODEL_DIR, 'onnx', 'vision_model.onnx')
example = torch.randn(1, 3, 224, 224)
kw = dict(input_names=['pixel_values'], output_names=['last_hidden_state', 'pooler_output'],
          dynamic_axes={'pixel_values': {0: 'batch_size'}, 'last_hidden_state': {0: 'batch_size'}, 'pooler_output': {0: 'batch_size'}},
          opset_version=17)
try:
    torch.onnx.export(export_model, (example,), onnx_path, dynamo=False, **kw)
except TypeError:
    torch.onnx.export(export_model, (example,), onnx_path, **kw)
onnx.checker.check_model(onnx_path)

# Check: ONNX gives the same vectors as PyTorch on the test photos.
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
x = torch.stack([ImageFiles([t['path']])[0] for t in tests[:8]])
with torch.no_grad(): ref = F.normalize(export_model(x)[1], dim=-1).numpy()
got = sess.run(['pooler_output'], {'pixel_values': x.numpy()})[0]
got = got / np.linalg.norm(got, axis=1, keepdims=True)
agree = float((ref * got).sum(1).min())
print(f'ONNX vs PyTorch agreement (1.0 = identical): {agree:.6f}')
assert agree > 0.9999, 'ONNX export does not match PyTorch'
shutil.make_archive(MODEL_DIR, 'zip', MODEL_DIR)
print('Saved', MODEL_DIR + '.zip', f'({os.path.getsize(MODEL_DIR + ".zip") / 2**20:.0f} MB)')

In [ ]:
# Final check with the exact library the scanner service runs (transformers.js 3.7.5 on Node).
# The two libraries resize images slightly differently, so even the current model does not match
# PyTorch exactly; the new model should match about as well as the current one does.
NODE_CHECK = 'not run'
if shutil.which('node') and shutil.which('npm'):
    jsdir = os.path.join('/tmp', 'tjs-check'); os.makedirs(jsdir, exist_ok=True)
    subprocess.run('npm init -y >/dev/null && npm install -s @huggingface/transformers@3.7.5 --onnxruntime-node-install-cuda=skip',
                   shell=True, cwd=jsdir, capture_output=True, text=True)
    script = f"""
import {{ AutoProcessor, SiglipVisionModel, RawImage, env }} from '@huggingface/transformers';
env.localModelPath = {json.dumps(OUT + '/')};
const image = await RawImage.read({json.dumps(tests[0]['path'])});
const res = [];
for (const id of ['siglip-bricksvault', {json.dumps(ONNX_TEMPLATE)}]) {{
  const p = await AutoProcessor.from_pretrained(id);
  const m = await SiglipVisionModel.from_pretrained(id, {{ dtype: 'fp32' }});
  res.push(Array.from((await m(await p(image))).pooler_output.data));
}}
console.log(JSON.stringify(res));
"""
    open(os.path.join(jsdir, 'check.mjs'), 'w').write(script)
    r = subprocess.run(['node', 'check.mjs'], cwd=jsdir, capture_output=True, text=True)
    if r.returncode == 0:
        new_js, base_js = (np.array(v) / np.linalg.norm(v) for v in json.loads(r.stdout.strip().splitlines()[-1]))
        x0 = ImageFiles([tests[0]['path']])[0][None]
        with torch.no_grad():
            base_ref = F.normalize(base_model.float().cpu()(pixel_values=x0).pooler_output, dim=-1)[0].numpy()
        new_ok, base_ok = float(new_js @ ref[0]), float(base_js @ base_ref)
        NODE_CHECK = f'new model {new_ok:.4f}, current model {base_ok:.4f}' + (' (OK)' if new_ok >= base_ok - 0.02 else ' (MISMATCH, tell Claude)')
    else:
        NODE_CHECK = 'failed: ' + r.stderr[-300:]
print('transformers.js vs PyTorch agreement:', NODE_CHECK)

In [ ]:
# Summary: copy everything between the lines into the BricksVault thread.
print('=' * 70)
print(summary)
print(f'ONNX check: {agree:.6f}; transformers.js check: {NODE_CHECK}')
print(f'Model saved to: {MODEL_DIR}.zip')
print('=' * 70)